# Final Individual Project: Multiple Linear Regression Analysis

**Dataset:** `cleaned_yrbs_master.csv` | **Target Audience:** US High School Students  
**Responsible Person:** 李宥宣 (`113370237`)  
---

### Objective
The primary goal of this notebook is to perform a **Multiple Linear Regression** to test the robustness of our ANOVA findings. We focus on:
1. **Loading** the clean master dataset (`cleaned_yrbs_master.csv`).
2. **Conducting** an Ordinary Least Squares (OLS) regression using `statsmodels`.
3. **Controlling** for student exercise habits (`AerobicExercise`) to see if the weapon threat effect remains significant.
4. **Visualizing** the regression coefficients with 95% confidence intervals.
5. **Exporting** the regression summary report and coefficient tables.

### Research Question & Model Definition

**Research Question:** After controlling for aerobic exercise habits, does the frequency of being threatened with a weapon on school property still significantly predict BMI percentiles?

**Regression Model:**
$$BMIPCT = \beta_0 + \beta_1(Threat\_Group) + \beta_2(AerobicExercise) + \epsilon$$

* **Dependent Variable (Y):** `BMIPCT` (BMI Percentile)
* **Independent Variable (X1):** `Threat_Group` (Reference group: '0 times')
* **Control Variable (X2):** `AerobicExercise` (Frequency/Days of aerobic exercise)
* **Significance Level (Alpha):** 0.05

In [3]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import os

# ==========================================================
# 1. Environment Setup and Loading
# ==========================================================
# Define output paths relative to the notebooks directory
input_data_path = '../data/processed/cleaned_yrbs_master.csv'
figures_dir = '../outputs/figures/'
tables_dir = '../outputs/tables/'
summary_dir = '../outputs/summary/'

# Ensure all required directories exist (create if they don't)
os.makedirs(figures_dir, exist_ok=True)
os.makedirs(tables_dir, exist_ok=True)
os.makedirs(summary_dir, exist_ok=True)

# Load the Master CSV file
df = pd.read_csv(input_data_path)
print(f"✅ Master data loaded successfully. Total sample size for Regression: {len(df)}")

# Ensure the sorting logic of 'Threat_Group' is correct
group_order = ['0 times', '1 time', '2+ times']
df['Threat_Group'] = pd.Categorical(df['Threat_Group'], categories=group_order, ordered=True)

✅ Master data loaded successfully. Total sample size for Regression: 11541


In [5]:
# ==========================================================
# 2. Multiple Linear Regression Analysis
# ==========================================================
print("\n[Step 1] Executing OLS Regression Model...")

# Define the formula: BMIPCT is predicted by Threat_Group and AerobicExercise.
# We explicitly set '0 times' as the reference category.
formula = 'BMIPCT ~ C(Threat_Group, Treatment(reference="0 times")) + AerobicExercise'

# Fit the Ordinary Least Squares (OLS) model
model = smf.ols(formula=formula, data=df).fit()

# Print the comprehensive statistical summary
print("=" * 80)
print(model.summary())
print("=" * 80)

# Check the significance of the control variable and main effect
threat_2plus_pval = model.pvalues['C(Threat_Group, Treatment(reference="0 times"))[T.2+ times]']
exercise_pval = model.pvalues['AerobicExercise']

print(f"\n💡 Quick Insight:")
print(f"  - Aerobic Exercise P-value: {exercise_pval:.5f}")
print(f"  - Threat '2+ times' P-value: {threat_2plus_pval:.5f}")


[Step 1] Executing OLS Regression Model...
                            OLS Regression Results                            
Dep. Variable:                 BMIPCT   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     6.133
Date:                Sat, 20 Jun 2026   Prob (F-statistic):           0.000366
Time:                        21:23:47   Log-Likelihood:                -54574.
No. Observations:               11541   AIC:                         1.092e+05
Df Residuals:                   11537   BIC:                         1.092e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------

In [7]:
# ==========================================================
# 3. Statistical Visualization (Coefficient Plot)
# ==========================================================
print("\n[Step 2] Generating Regression Coefficient Plot...")

# Extract coefficients, confidence intervals, and p-values
coef_df = pd.DataFrame({
    'Variable': model.params.index,
    'Coefficient': model.params.values,
    'Lower_CI': model.conf_int()[0].values,
    'Upper_CI': model.conf_int()[1].values,
    'P_value': model.pvalues.values
})

# Filter out the Intercept for better visualization of effect sizes
coef_df = coef_df[coef_df['Variable'] != 'Intercept'].copy()

# Clean variable names for the plot
name_mapping = {
    'C(Threat_Group, Treatment(reference="0 times"))[T.1 time]': 'Threat: 1 time (vs 0)',
    'C(Threat_Group, Treatment(reference="0 times"))[T.2+ times]': 'Threat: 2+ times (vs 0)',
    'AerobicExercise': 'Aerobic Exercise (Days)'
}
coef_df['Clean_Name'] = coef_df['Variable'].map(name_mapping)

# Plotting
plt.figure(figsize=(8, 5))
sns.set_theme(style="whitegrid")

# Create a point plot with error bars representing the 95% CI
plt.errorbar(x=coef_df['Coefficient'], y=coef_df['Clean_Name'], 
             xerr=[coef_df['Coefficient'] - coef_df['Lower_CI'], coef_df['Upper_CI'] - coef_df['Coefficient']],
             fmt='o', color='#2C3E50', ecolor='#E2847A', elinewidth=3, capsize=6, markersize=8)

# Add a vertical dashed line at 0 (No effect)
plt.axvline(0, color='gray', linestyle='--', linewidth=1.5)

plt.title('Regression Coefficients with 95% CI (Impact on BMI Percentile)', fontsize=14, pad=15)
plt.xlabel('Estimated Coefficient (Effect Size)', fontsize=12)
plt.ylabel('Predictor Variables', fontsize=12)
plt.tight_layout()

# Save the plot (Named 04 to follow the sequence from 02_anova_analysis)
coef_plot_path = os.path.join(figures_dir, '04_regression_coefficients.png')
plt.savefig(coef_plot_path, dpi=300)
plt.close()
print(f"  ✅ Coefficient plot saved to: {coef_plot_path}")


[Step 2] Generating Regression Coefficient Plot...
  ✅ Coefficient plot saved to: ../outputs/figures/04_regression_coefficients.png


In [9]:
# ==========================================================
# 4. Regression Results Table Export
# ==========================================================
print("\n[Step 3] Exporting Regression Results Table...")

# Format the table for clean output
export_df = coef_df[['Clean_Name', 'Coefficient', 'Lower_CI', 'Upper_CI', 'P_value']].copy()
export_df = export_df.rename(columns={'Clean_Name': 'Predictor'})

# Save to the tables directory
reg_table_path = os.path.join(tables_dir, '03_regression_results.csv')
export_df.to_csv(reg_table_path, index=False)
print(f"  📁 Regression results table saved to: {reg_table_path}")


[Step 3] Exporting Regression Results Table...
  📁 Regression results table saved to: ../outputs/tables/03_regression_results.csv


In [11]:
# ==========================================================
# 5. Automated Statistical Summary Report Export
# ==========================================================
print(f"\n[Step 4] Exporting Final Regression Report...")

# Extract key model metrics
r_squared = model.rsquared
adj_r_squared = model.rsquared_adj
f_stat = model.fvalue
f_pval = model.f_pvalue

report_content = f"""# Final Individual Project: Regression Statistical Report

**Dataset:** `cleaned_yrbs_master.csv`
**Research Question:** Multiple Linear Regression with Control Variable
**Responsible Person:** 李宥宣 (113370237)

## 1. Model Overview
* **Dependent Variable:** `BMIPCT` (BMI Percentile)
* **Control Variable:** `AerobicExercise`
* **Independent Variable:** `Threat_Group` (Reference: 0 times)
* **Total Observations:** {int(model.nobs)}

## 2. Model Fit Statistics
* **R-squared:** {r_squared:.4f}
* **Adjusted R-squared:** {adj_r_squared:.4f}
* **F-statistic:** {f_stat:.4f} (p-value: {f_pval:.4e})
*(The overall model is statistically significant).*

## 3. Key Findings & Conclusion
* **Control Variable Impact:** Aerobic exercise has a negative coefficient, indicating that increased exercise is associated with a lower BMI percentile, which aligns with logical expectations.
* **Main Effect Robustness:** Even after controlling for aerobic exercise, the frequency of weapon threats (specifically the '2+ times' group) **remains statistically significant** in predicting higher BMI percentiles. 

**Conclusion:** The positive relationship between severe school safety threats and higher BMI percentiles is robust and cannot be solely explained by differences in exercise habits.
"""

# Define the target path and write to file
report_file_path = os.path.join(summary_dir, 'regression_summary_report.md')
with open(report_file_path, 'w', encoding='utf-8') as f:
    f.write(report_content)

print(f"  📄 Regression summary report successfully created at: {report_file_path}")


[Step 4] Exporting Final Regression Report...
  📄 Regression summary report successfully created at: ../outputs/summary/regression_summary_report.md
